In [3]:
import opendatasets as od
import os

UCI_Poker = 'https://www.kaggle.com/datasets/rasvob/uci-poker-hand-dataset/data?select=poker-hand-testing.data'
od.download(UCI_Poker)

os.listdir('./uci-poker-hand-dataset')

Skipping, found downloaded files in ".\uci-poker-hand-dataset" (use force=True to force download)


['poker-hand-testing.data', 'poker-hand-training-true.data']

In [8]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

data_train = pd.read_csv(filepath_or_buffer='./uci-poker-hand-dataset/poker-hand-training-true.data', sep=',', header=None)
data_test = pd.read_csv(filepath_or_buffer='./uci-poker-hand-dataset/poker-hand-testing.data', sep=',', header=None)

X_train = data_train.values[:,0:10] # first 10 columns
y_train = data_train.values[:,10]   # 11th column

X_test = data_test.values[:,0:10]
y_test = data_test.values[:,10]

# Hyperparameter tuning
params = {
    'n_estimators': [100, 200],          # number of trees
    'max_depth': [5, 10, 20],            # depth of each tree
    'min_samples_split': [5, 10],        # split threshold
    'min_samples_leaf': [2, 4],          # leaf size
    'max_features': ['sqrt', 'log2']     # features per split   
}

model = RandomForestClassifier(random_state=123)

print("Starting tuning...")
grid = GridSearchCV(model, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Best set of hyperparameters: ", grid.best_params_)
print("Best score: ", grid.best_score_)

Starting tuning...
Best set of hyperparameters:  {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best score:  0.6143942423030788


In [9]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Use the best model
best_model = grid.best_estimator_

# Predictions
train_preds = best_model.predict(X_train)
test_preds = best_model.predict(X_test)

# Accuracy
accuracy_train = accuracy_score(y_train, train_preds)
accuracy_test = accuracy_score(y_test, test_preds)
print(f"Train Accuracy: {accuracy_train:.2f}")
print(f"Test Accuracy: {accuracy_test:.2f}\n")

# F1 Score (weighted for multi-class)
f1_train = f1_score(y_train, train_preds, average="weighted")
f1_test = f1_score(y_test, test_preds, average="weighted")
print(f"F1 Score (train): {f1_train:.2f}")
print(f"F1 Score (test): {f1_test:.2f}\n")

# Confusion matrix and classification report
print("Confusion Matrix (Test):")
print(confusion_matrix(y_test, test_preds))
print("\nClassification Report (Test):")
print(classification_report(y_test, test_preds, digits=3, zero_division=0))

Train Accuracy: 0.95
Test Accuracy: 0.62

F1 Score (train): 0.93
F1 Score (test): 0.58

Confusion Matrix (Test):
[[415105  86104      0      0      0      0      0      0      0      0]
 [221348 201138     12      0      0      0      0      0      0      0]
 [ 12654  34952     16      0      0      0      0      0      0      0]
 [  3399  17717      4      1      0      0      0      0      0      0]
 [   154   3731      0      0      0      0      0      0      0      0]
 [  1824    172      0      0      0      0      0      0      0      0]
 [    50   1370      4      0      0      0      0      0      0      0]
 [     3    227      0      0      0      0      0      0      0      0]
 [     3      9      0      0      0      0      0      0      0      0]
 [     1      2      0      0      0      0      0      0      0      0]]

Classification Report (Test):
              precision    recall  f1-score   support

           0      0.634     0.828     0.718    501209
           1    

The Random Forest model achieved high training performance with a train accuracy of 0.95 and weighted F1 score of 0.93, indicating it learned the patterns in the training data very well. On the test set, accuracy dropped to 0.62 and weighted F1 score to 0.58, showing some overfitting and the challenge of predicting rare poker hand classes. The confusion matrix and classification report reveal that the model predicts the most common hands (classes 0 and 1) reasonably well, but performs poorly on rare hands, which is expected given the extreme class imbalance in this dataset.